In [0]:
!pip install pandas

In [0]:
#Importing required Libraries
import pandas as pd
import re
import numpy as np

In [0]:
#Import the dataset cc_calls and billings
cc_calls=pd.read_csv("Dataset/cc_calls.csv",low_memory=False)
billing_df=pd.read_csv("Dataset/billings.csv",low_memory=False)

## Filtering the Billings data

In [0]:
# Removing duplicate Co_Ref entries while keeping the newest renewal date
billing_df['Co_Ref'].value_counts().head()
billing_clean = billing_df.sort_values('Prospect_Renewal_Date').drop_duplicates(
    'Co_Ref', keep='last'
)
len(billing_clean)

In [0]:
billing_clean['Prospect_Renewal_Date'].isna().sum()

In [0]:
#Joining the Billings and cc_calls with respect to co_Ref
df_calls_merged = cc_calls.merge(
    billing_clean[['Co_Ref', 'Prospect_Renewal_Date']],
    on='Co_Ref',
    how='inner'
)
len(df_calls_merged)

# Exploratory Data Analysis (EDA)

In [0]:
df_calls_merged.info()
len(df_calls_merged)

In [0]:
df_calls_merged.describe()

In [0]:
df_calls_merged.isna().sum()

In [0]:
# Fix data types for date columns
date_cols = [
    "Call_Date",
    "Prospect_Renewal_Date",
]
for col in date_cols:
    if col in df_calls_merged.columns:
        parsed = pd.to_datetime(
            df_calls_merged[col],
            dayfirst=True,
        )
        df_calls_merged[col] = parsed.combine_first(df_calls_merged[col])

# convert Contact_ID and Co_Ref to string
df_calls_merged['Contact_ID'] = df_calls_merged['Contact_ID'].astype('Int64').astype('string')
df_calls_merged['Co_Ref'] = df_calls_merged['Co_Ref'].astype('string')

# Convert all object-type columns into category type
cat_cols = df_calls_merged.select_dtypes(include='object').columns
df_calls_merged[cat_cols] = df_calls_merged[cat_cols].astype('category')

In [0]:
#Convert score values to numerical values
score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

df_calls_merged[score_cols] = df_calls_merged[score_cols].apply(
    pd.to_numeric, errors='coerce'
)

## Replacing Null Values

In [0]:
# 1. Handle categorical columns
for col in cat_cols:
    if "Unknown" not in df_calls_merged[col].cat.categories:
        df_calls_merged[col] = df_calls_merged[col].cat.add_categories(["Unknown"])
    df_calls_merged[col] = df_calls_merged[col].fillna("Unknown")

In [0]:
# 2. Handle numeric score columns
score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]
df_calls_merged[score_cols] = df_calls_merged[score_cols].fillna(
    df_calls_merged[score_cols].fillna(-1)
)

In [0]:
# drop rows that have Co_Ref=NULL
df_calls_merged['Co_Ref'].isna().sum()
df_calls_merged = df_calls_merged.dropna(subset=['Co_Ref'])

In [0]:
yes_no_cols = [
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact', 
    'cc_refund_discussed', 'cc_contractor_suggest_leave', 
    'cc_contractor_complained'
]
for col in yes_no_cols:
    df_calls_merged[col] = df_calls_merged[col].apply(lambda x: 'No' if str(x).lower().strip() in ['no', '', 'nan', 'none','not'] else 'Yes')
    df_calls_merged[col]= df_calls_merged[col].fillna('Unknown')

## Remove Unnecessary Column

In [0]:
df_calls_merged = df_calls_merged.drop(columns=['cc_contractor_sentiment'])
df_calls_merged = df_calls_merged.drop(columns=['Analysed_Call'])

In [0]:
df_calls_merged.info()

In [0]:
#Do sentiment analysis for the fields that contain descriptive sentence
def clean_messy_binary(val):
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown"
    
    text = str(val).strip().lower()

    # explicit yes/no
    if text in ["yes", "y", "true", "1"]:
        return "Yes"
    if text in ["no", "n", "false", "0", "[yes/no]"]:
        return "No"

    if len(text.split()) > 3:
        return "Yes"

    # if contains helpful/action words
    keywords_yes = [
        "help", "support", "assist", "review", "imply", "process",
        "accreditation", "contractor", "updated", "investigated"
    ]
    if any(k in text for k in keywords_yes):
        return "Yes"
    #If it is digits convert that to "Yes" 
    if re.fullmatch(r"\d+", text):
        return "Yes"

    return "Unknown"


In [0]:
#Converting the messy columns to yes/no 
messy_cols = [
    "cc_agent_cross_sell_attempt",
    "cc_customer_issues_concerns",
    "cc_business_struggles_financial_hardship",
    "cc_chasing_response",
    "cc_issues_within_questionnaire",
    "cc_login_issues",
    "cc_platform_issues",
    "cc_process_complexity_concerns",
    "cc_questions_harder_than_expected",
    "cc_dissatisfaction_support",
    ]

for col in messy_cols:
    df_calls_merged[col] = df_calls_merged[col].apply(clean_messy_binary).astype("category")

In [0]:
for col in messy_cols:
    print(col, df_calls_merged[col].value_counts(dropna=False))

In [0]:
df_calls_merged.to_csv('cc_calls_cleaned.csv', index=False)

# Applying Filter

In [0]:
# Apply filter according to the prospect_renewal_date (Post Renewal churn)
df_calls_merged = df_calls_merged[
    df_calls_merged['Call_Date'] > df_calls_merged['Prospect_Renewal_Date']
]

In [0]:
df_calls_merged.to_csv('cc_calls_cleaned_filtered.csv', index=False)